# BATS and TBATS: Forecasting Complex Seasonality

BATS and TBATS fit a level, a damped trend, ARMA errors and several seasonal cycles in one state space model, after an optional Box-Cox transformation ([De Livera, Hyndman and Snyder (2011)](https://doi.org/10.1198/jasa.2011.tm09771)). This tutorial simulates a series with one cycle and a series with two, fits both models, reads what they chose, and scores them against a seasonal naive forecast on held-back data.

In [ ]:
Pollis.packages("Durbyn", "Plots")

In [ ]:
using Durbyn, Random, Statistics
import Plots

Random.seed!(42)

mase(actual, predicted, train) = mean(abs.(actual .- predicted)) / mean(abs.(diff(train)))

## One Seasonal Cycle

The first series is monthly in spirit: a trend, a cycle of 12 and noise. The last 24 values are held back.

In [ ]:
n = 144
tt = 1:n
y = 100 .+ 0.4 .* tt .+ 12 .* sin.(2pi .* tt ./ 12) .+ 3 .* randn(n)
train, test = y[1:end-24], y[end-23:end]
Plots.plot(y, label="series", xlabel="period", title="One seasonal cycle")

`bats` keeps one seasonal state per month, and `tbats` writes the cycle as sine and cosine terms. Both try the Box-Cox transformation, the trend and the damping, and keep the combination with the lowest AIC. `lambda` is `nothing` when no transformation was kept.

In [ ]:
fit_bats = bats(train, [12])
fit_tbats = tbats(train, [12])
for (name, model) in (("BATS", fit_bats), ("TBATS", fit_tbats))
    println(name, ": AIC ", round(model.aic, digits=1), ", lambda ", model.lambda, ", damping ", model.damping_parameter)
end

## Scoring the Forecasts on the Holdout

MASE divides the mean absolute error by the in-sample error of the one-step naive forecast, and a value under 1 beats that forecast ([Hyndman and Koehler (2006)](https://doi.org/10.1016/j.ijforecast.2006.03.001)). A seasonal naive forecast repeats the last observed cycle, and is the benchmark to beat here.

In [ ]:
h = length(test)
fc_bats = forecast(fit_bats; h=h)
fc_tbats = forecast(fit_tbats; h=h)
fc_snaive = forecast(snaive(train, 12); h=h)
for (name, fc) in (("BATS", fc_bats), ("TBATS", fc_tbats), ("Seasonal naive", fc_snaive))
    println(rpad(name, 16), "MASE ", round(mase(test, fc.mean, train), digits=3))
end

In [ ]:
Plots.plot(length(train)-47:length(train), train[end-47:end], label="train", xlabel="period", title="TBATS forecast with 95% interval")
Plots.plot!(length(train)+1:n, test, label="held back")
Plots.plot!(length(train)+1:n, fc_tbats.mean, ribbon=(fc_tbats.mean .- fc_tbats.lower[:, 2], fc_tbats.upper[:, 2] .- fc_tbats.mean), label="TBATS")

## Two Seasonal Cycles and a Non-Integer Period

Hourly data often has a daily cycle of 24 and a weekly cycle of 168. BATS would need 192 seasonal states for those two cycles, so TBATS is the practical choice. The second series has both cycles over ten weeks, of which the last week is held back.

In [ ]:
n2 = 24 * 7 * 10
t2 = 1:n2
y2 = 50 .+ 8 .* sin.(2pi .* t2 ./ 24) .+ 5 .* sin.(2pi .* t2 ./ 168) .+ 2 .* randn(n2)
train2, test2 = y2[1:end-168], y2[end-167:end]
fit_two = tbats(train2, [24, 168])
fc_two = forecast(fit_two; h=168)
println("TBATS ", round(mase(test2, fc_two.mean, train2), digits=3))
println("Seasonal naive ", round(mase(test2, forecast(snaive(train2, 168); h=168).mean, train2), digits=3))

In [ ]:
Plots.plot(test2, label="held back", xlabel="hour", title="Two seasonal cycles")
Plots.plot!(fc_two.mean, label="TBATS", lw=2)

A non-integer period goes in the same way: `tbats(y, [52.18])` fits weekly data with a yearly cycle, which `bats` refuses as its periods must be whole numbers.

## What to Take from These Fits

The estimated structure depends on the series, and rerunning with another seed can change which components are kept. Judge the model by its holdout score against the seasonal naive forecast, then read its parameters. Long or many periods make the fit slower, because more harmonics are tried.

## References

De Livera, A. M., Hyndman, R. J. and Snyder, R. D. (2011). Forecasting time series with complex seasonal patterns using exponential smoothing. *Journal of the American Statistical Association* 106(496), 1513–1527. [doi:10.1198/jasa.2011.tm09771](https://doi.org/10.1198/jasa.2011.tm09771)

Hyndman, R. J. and Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting* 22(4), 679–688. [doi:10.1016/j.ijforecast.2006.03.001](https://doi.org/10.1016/j.ijforecast.2006.03.001)